# Curated Forecast & Portfolio Analytics Workbook

This workbook was generated from the uploaded CSV collection and is designed to be **client-presentable**, not just exploratory.

## What this workbook does
- Catalogs all uploaded datasets and recommends the best analytical use case for each.
- Selects the strongest forecasting candidate from the collection.
- Builds a **30-business-day GLD forecast** using lag and rolling-window features.
- Benchmarks multiple models using a chronological holdout.
- Adds supporting insight panels from **BigMart** and **Insurance** to show how the broader portfolio can support retail and pricing analytics.

## Current project summary
- **Datasets profiled:** 18
- **Total rows:** 368,143
- **Best forecast model:** Ridge
- **Backtest RMSE:** 0.979
- **History window:** 2008-01-02 to 2018-05-16
- **Latest actual GLD:** 122.54
- **30-day forecast endpoint:** 120.74

> Why GLD for the forecast? Among the uploaded files, `gld_price_data.csv` is the cleanest time-series dataset with a true date key and continuous target values. That makes it the most defensible choice for a forecasting deliverable.


In [ ]:

from pathlib import Path
import math
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base = Path('/mnt/data')
out_dir = base / 'curated_forecast_project'
out_dir.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)


## 1) Dataset catalog and project framing

In [ ]:

dataset_configs = {
    'bigmart_data.csv': {'domain':'Retail', 'recommended_problem':'Sales regression / assortment insights', 'primary_target':'Item_Outlet_Sales'},
    'breast_data.csv': {'domain':'Healthcare', 'recommended_problem':'Cancer diagnosis classification', 'primary_target':'diagnosis'},
    'calories.csv': {'domain':'Health & Fitness', 'recommended_problem':'Calories regression (after joining exercise)', 'primary_target':'Calories'},
    'CAR DETAILS FROM CAR DEKHO.csv': {'domain':'Automotive', 'recommended_problem':'Used-car price regression', 'primary_target':'selling_price'},
    'creditcard.csv': {'domain':'FinTech', 'recommended_problem':'Fraud classification / anomaly detection', 'primary_target':'Class'},
    'diabetes.csv': {'domain':'Healthcare', 'recommended_problem':'Diabetes classification', 'primary_target':'Outcome'},
    'exercise.csv': {'domain':'Health & Fitness', 'recommended_problem':'Workout analytics / feature engineering', 'primary_target':'Calories (joined)'},
    'FakeNewsNet.csv': {'domain':'Media', 'recommended_problem':'Fake news classification', 'primary_target':'real'},
    'gender_submission.csv': {'domain':'Benchmark', 'recommended_problem':'Binary classification benchmark', 'primary_target':'Survived'},
    'gld_price_data.csv': {'domain':'Markets', 'recommended_problem':'Time-series forecasting', 'primary_target':'GLD'},
    'heart_disease_data.csv': {'domain':'Healthcare', 'recommended_problem':'Heart disease classification', 'primary_target':'target'},
    'insurance.csv': {'domain':'Insurance', 'recommended_problem':'Charge regression / pricing analytics', 'primary_target':'charges'},
    'loan.csv': {'domain':'Banking', 'recommended_problem':'Loan approval classification', 'primary_target':'Loan_Status'},
    'mail_data.csv': {'domain':'NLP', 'recommended_problem':'Spam classification', 'primary_target':'Category'},
    'Mall_Customers.csv': {'domain':'Retail', 'recommended_problem':'Customer segmentation / clustering', 'primary_target':'Spending Score (1-100)'},
    'movies.csv': {'domain':'Entertainment', 'recommended_problem':'Popularity / revenue modeling', 'primary_target':'revenue'},
    'parkinson.csv': {'domain':'Healthcare', 'recommended_problem':'Parkinson classification', 'primary_target':'status'},
    'sonar_data.csv': {'domain':'Defense', 'recommended_problem':'Rock-vs-mine classification', 'primary_target':'last_column_label'},
}

catalog_rows = []
for file_name, meta in dataset_configs.items():
    df = pd.read_csv(base / file_name)
    catalog_rows.append({
        'dataset': file_name,
        'rows': int(df.shape[0]),
        'columns': int(df.shape[1]),
        'missing_cells': int(df.isna().sum().sum()),
        'numeric_columns': int(df.select_dtypes(include=np.number).shape[1]),
        'non_numeric_columns': int(df.select_dtypes(exclude=np.number).shape[1]),
        **meta
    })

catalog = pd.DataFrame(catalog_rows).sort_values(['domain', 'dataset']).reset_index(drop=True)
catalog


In [ ]:

fig = px.scatter(
    catalog, x='rows', y='columns', size='missing_cells', color='domain',
    hover_name='dataset', log_x=True,
    title='Dataset Portfolio Map: Scale vs Feature Breadth'
)
fig.update_layout(height=450)
fig.show()


## 2) GLD forecasting candidate deep dive

This section creates a forecasting-ready feature set from the `GLD` series:
- lags: 1, 2, 3, 5, 10, 20
- rolling means and rolling standard deviations
- calendar features: month, quarter, weekday, year
- a simple trend index

The holdout is **chronological** so the evaluation simulates real forecasting conditions.


In [ ]:

gld = pd.read_csv(base / 'gld_price_data.csv')
gld['Date'] = pd.to_datetime(gld['Date'])
gld = gld.sort_values('Date').reset_index(drop=True)

feat_df = gld.copy()
for lag in [1, 2, 3, 5, 10, 20]:
    feat_df[f'lag_{lag}'] = feat_df['GLD'].shift(lag)
for window in [5, 10, 20]:
    feat_df[f'roll_mean_{window}'] = feat_df['GLD'].shift(1).rolling(window).mean()
    feat_df[f'roll_std_{window}'] = feat_df['GLD'].shift(1).rolling(window).std()

feat_df['month'] = feat_df['Date'].dt.month
feat_df['quarter'] = feat_df['Date'].dt.quarter
feat_df['dayofweek'] = feat_df['Date'].dt.dayofweek
feat_df['year'] = feat_df['Date'].dt.year
feat_df['trend'] = np.arange(len(feat_df))

model_df = feat_df.dropna().reset_index(drop=True)
ts_features = [c for c in model_df.columns if c not in ['Date','GLD','SPX','USO','SLV','EUR/USD']]

X = model_df[ts_features]
y = model_df['GLD']

split_idx = int(len(model_df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
date_test = model_df['Date'].iloc[split_idx:]

models = {
    'Ridge': Pipeline([('imputer', SimpleImputer()), ('model', Ridge(alpha=1.0))]),
    'LinearRegression': Pipeline([('imputer', SimpleImputer()), ('model', LinearRegression())]),
    'RandomForest': Pipeline([('imputer', SimpleImputer()), ('model', RandomForestRegressor(
        n_estimators=250, random_state=42, n_jobs=-1, max_depth=8, min_samples_leaf=2
    ))]),
}

bench = []
fitted_models = {}
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = pred
    bench.append({
        'model': name,
        'rmse': float(np.sqrt(mean_squared_error(y_test, pred))),
        'mae': float(mean_absolute_error(y_test, pred)),
        'mape_pct': float(np.mean(np.abs((y_test - pred) / y_test)) * 100),
        'r2': float(r2_score(y_test, pred))
    })

bench_df = pd.DataFrame(bench).sort_values('rmse').reset_index(drop=True)
bench_df


In [ ]:

fig = px.bar(
    bench_df.sort_values('rmse'),
    x='model', y=['rmse', 'mae'],
    barmode='group',
    title='GLD Forecast Model Benchmark (lower is better)'
)
fig.update_layout(height=420)
fig.show()


In [ ]:

best_model_name = bench_df.iloc[0]['model']
best_model = fitted_models[best_model_name]
best_test_pred = predictions[best_model_name]
residuals = y_test.to_numpy() - best_test_pred
resid_std = float(np.std(residuals))

# Refit on full observed history for final forward forecast
best_model.fit(X, y)

history = gld[['Date', 'GLD']].copy()
future_dates = pd.bdate_range(history['Date'].max() + pd.Timedelta(days=1), periods=30)

working = history.copy()
forecast_rows = []

for step, fdate in enumerate(future_dates, start=1):
    glist = working['GLD'].tolist()
    row = {'Date': fdate}
    for lag in [1, 2, 3, 5, 10, 20]:
        row[f'lag_{lag}'] = glist[-lag]
    for window in [5, 10, 20]:
        arr = np.array(glist[-window:])
        row[f'roll_mean_{window}'] = float(arr.mean())
        row[f'roll_std_{window}'] = float(arr.std(ddof=1)) if len(arr) > 1 else 0.0
    row['month'] = fdate.month
    row['quarter'] = (fdate.month - 1) // 3 + 1
    row['dayofweek'] = fdate.dayofweek
    row['year'] = fdate.year
    row['trend'] = len(working)

    row_df = pd.DataFrame([row])[ts_features]
    yhat = float(best_model.predict(row_df)[0])
    band = resid_std * math.sqrt(1 + step / 10)

    forecast_rows.append({
        'Date': fdate,
        'forecast_gld': yhat,
        'lower_95': yhat - 1.96 * band,
        'upper_95': yhat + 1.96 * band
    })
    working = pd.concat([working, pd.DataFrame([{'Date': fdate, 'GLD': yhat}])], ignore_index=True)

forecast_df = pd.DataFrame(forecast_rows)
forecast_df.head()


In [ ]:

test_pred_df = pd.DataFrame({
    'Date': date_test.values,
    'actual_gld': y_test.values,
    'predicted_gld': best_test_pred
})

recent_hist = gld[gld['Date'] >= gld['Date'].max() - pd.Timedelta(days=700)].copy()

fig = go.Figure()
fig.add_trace(go.Scatter(x=recent_hist['Date'], y=recent_hist['GLD'], mode='lines', name='Historical GLD'))
fig.add_trace(go.Scatter(x=test_pred_df['Date'], y=test_pred_df['predicted_gld'], mode='lines', name='Backtest Prediction'))
fig.add_trace(go.Scatter(x=forecast_df['Date'], y=forecast_df['upper_95'], mode='lines', line=dict(width=0), hoverinfo='skip', showlegend=False))
fig.add_trace(go.Scatter(
    x=forecast_df['Date'], y=forecast_df['lower_95'], mode='lines',
    fill='tonexty', fillcolor='rgba(99,110,250,0.15)', line=dict(width=0),
    name='95% interval', hoverinfo='skip'
))
fig.add_trace(go.Scatter(x=forecast_df['Date'], y=forecast_df['forecast_gld'], mode='lines+markers', name='30-business-day Forecast'))
fig.update_layout(
    title=f'GLD Forecast Deep Dive — Best Model: {best_model_name}',
    height=500, xaxis_title='Date', yaxis_title='GLD'
)
fig.show()

forecast_df.tail(10)


### Interpretation
- The best model here is **feature-based autoregression**, not a black-box deep model.
- That is actually useful for client delivery: it is lighter to maintain, easier to explain, and performs well on the holdout.
- The uncertainty band widens through the 30-business-day horizon to reflect compounding forecast error.


## 3) Supporting analytics from the rest of the dataset portfolio

In [ ]:

bigmart = pd.read_csv(base / 'bigmart_data.csv')
bigmart['Outlet_Size'] = bigmart['Outlet_Size'].fillna('Unknown')
bigmart['Item_Weight'] = bigmart['Item_Weight'].fillna(bigmart['Item_Weight'].median())

sales_by_outlet = (bigmart.groupby('Outlet_Type', as_index=False)['Item_Outlet_Sales']
                   .mean()
                   .sort_values('Item_Outlet_Sales', ascending=False))

fig = px.bar(
    sales_by_outlet, x='Item_Outlet_Sales', y='Outlet_Type', orientation='h',
    title='BigMart — Average Sales by Outlet Type'
)
fig.update_layout(height=400)
fig.show()

sales_by_outlet


In [ ]:

insurance = pd.read_csv(base / 'insurance.csv')

fig = px.box(
    insurance, x='smoker', y='charges', color='smoker',
    title='Insurance — Charges by Smoking Status',
    points='outliers'
)
fig.update_layout(height=420)
fig.show()

insurance.groupby('smoker')['charges'].agg(['mean', 'median', 'count']).reset_index()


## 4) Recommended client-facing storyline

### Core project to sell
**Data + AI Portfolio Accelerator with Forecasting Dashboard**

### Phase 1
- Ingest and catalog source data.
- Build quality checks and a profiling layer.

### Phase 2
- Create curated analytical workstreams:
  - forecasting from time-series data
  - regression from structured commercial data
  - classification from operational or medical data
  - segmentation from customer data

### Phase 3
- Publish dashboards:
  - executive KPI dashboard
  - forecasting monitor
  - model benchmark board
  - data quality scorecard

### Why this workbook is commercially strong
It proves you can move from **raw CSVs → data catalog → model comparison → forecast output → dashboard-ready assets** in one coherent delivery.


In [ ]:

# Persist reusable outputs so they can be fed into Power BI or another dashboarding layer.
catalog.to_csv(out_dir / 'dataset_catalog.csv', index=False)
bench_df.to_csv(out_dir / 'gld_model_benchmark.csv', index=False)
test_pred_df.to_csv(out_dir / 'gld_test_predictions.csv', index=False)
forecast_df.to_csv(out_dir / 'gld_30_business_day_forecast.csv', index=False)

print('Saved reusable outputs to:', out_dir)
print('\nGenerated files:')
for path in sorted(out_dir.iterdir()):
    print('-', path.name)
